In [1]:
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as st
from scipy.stats import describe
import missingno as msno

In [2]:
# FUNCIONES AUXILIARES
def plot_histograma(data, column, figsize=(6, 3), bins=15, kde=True, mvd=True, shade=True, snk=False):
    skewness = (data[column]).skew()
    kurtosis = (data[column]).kurt()
    media = (data[column]).mean()
    var = (data[column]).var()
    std = (data[column]).std()
    
    plt.figure(figsize=figsize)
    plt.grid(axis='y')
    sns.histplot(data[column], bins=bins, kde=kde)
    
    if snk:
        plt.figtext(0.7, 0.8, f'Asimetría: {skewness:.2f}', fontsize=10, color='blue')
        plt.figtext(0.715, 0.73, f'Curtosis: {kurtosis:.2f}', fontsize=10, color='blue')
        plt.axvline(media, color='red', linestyle='--', label='Media')

    if mvd and shade:
        plt.axvspan(media - std, media + std, alpha=0.1, color='orange', label='±1 Std')
        plt.axvline(media + std, color='orange', linestyle=':', label='+1 Std')
        plt.axvline(media - std, color='orange', linestyle=':', label='-1 Std')
        plt.figtext(0.15, 0.80, f'Media: {media:.2f}', fontsize=10, color='red')
        plt.figtext(0.15, 0.73, f'Var:   {var:.2f}',   fontsize=10, color='red')
        plt.figtext(0.15, 0.66, f'Std:   {std:.2f}',   fontsize=10, color='orange')

    plt.title(f'Variable: {column}')
    plt.xlabel(f'{column}')
    plt.ylabel('Frecuencia')
    plt.show()

1. ENSEÑARLE A LA COMPUTADORA

como quiero que la computadora empiece a predecir el futuro , necesito seguir un proceso lógico.

Primero voy a separar X de y 
X: Pistas que el modelo usa para analizar la situacion , como rival , posicion , minutos de partido,etc.
y: ES la respuesta final que queremos predecir 
Separarlos enseña a la maquina.

In [ ]:
np.random.seed(42)
n_rows = 100

data = {
    'Opponent': np.random.choice(['Real Madrid', 'Atletico Madrid', 'Sevilla', 'Eibar', 'Getafe'], size=n_rows),
    'Playing_Position': np.random.choice(['RW', 'CF', 'CAM', np.nan], size=n_rows, p=[0.45, 0.40, 0.10, 0.05]), # 5% nulos
    'Minute': np.random.choice([12, 35, 45, 68, 88, 90, 120, 350], size=n_rows, p=[0.15, 0.15, 0.15, 0.20, 0.20, 0.10, 0.03, 0.02]), # Outliers (350 min)
    'Competition': np.random.choice(['LaLiga', 'UCL', 'Copa del Rey'], size=n_rows, p=[0.6, 0.25, 0.15]),
    'Shot_Distance_m': np.random.choice([5, 12, 18, 25, 60, np.nan], size=n_rows, p=[0.3, 0.3, 0.2, 0.1, 0.02, 0.08]), # Nulos y Outliers
    'Goal_Scored': np.random.choice([1, 0], size=n_rows, p=[0.65, 0.35]) # Variable Target
}

df_messi = pd.DataFrame(data)

print("Dataset creado, Filas y columnas: ", df_messi)


OBSERVACION IMPORTANTE 

Para saber cual es Y y que dejar en X , podemos aplicar 2 preguntas clave:

¿Cual es el objetivo? --> mi objetivo es predecir si una jugada termina en gol , 
entonces es la columna 'Goal_Scored'

¿Tengo la info dispiible antes de que ocurra el evento?
Distancia del tiro , rival , posicion se conocen en el instante previo a patear , entonces eso va a X

In [ ]:
X = df_messi.drop(columns=['Goal_Scored'])

y= df_messi['Goal_Scored']

print("Dimensiones de las pistas (X)" , X.shape)
print("Dimensiones de las respuestas (y)" , y.shape)


Realizar TRAIN VS TEST SPLIT
Una vez separado X e y , dividimos el dataset para guardar un 20%

In [ ]:
from sklearn.model_selection import train_test_split

# Dividimos 80% entrenamiento y 20% prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,     # 20% para test
    random_state=42,    # Reproducibilidad
    stratify=y          # Mantiene la misma proporcion de goles en train y test
)

print(f"Filas de entrenamiento (Train): {len(X_train)}")
print(f"Filas de evaluación (Test):    {len(X_test)}")

TRATAMIENTO DE DATOS FALTANTES (NULOS)

In [ ]:
#Primero quiero identificar cuantos nulos cayeron dentro del conjunto entrenamiento
print("valores nulos por columna en X-train")

print(X_train.isna().sum())

Una vez que se que no tengo datos nulos , debo DEFINIR UNA REGLA DE RELLENO CON LOS DATOS DE ENTRENAMIENTO 
y aplicarla para tapar los huecos vacios.

Redefinimos en x e y , luego verificamos los datos nulos , los datos nulos que obtengamos se vana rellenar automaticamente con los valores de la distancia y posicion calculada por la medaina y moda que son los valores mas tipicos

In [ ]:
# Calculamos la mediana y la moda SOLO sobre X_train
mediana_distancia = X_train['Shot_Distance_m'].median()
moda_posicion = X_train['Playing_Position'].mode()[0]

print(f"Mediana calculada para distancia: {mediana_distancia}m")
print(f"Moda calculada para posición: {moda_posicion}")

#  Imputamos X_train y X_test usando los valores obtenidos de train
X_train_clean = X_train.copy()
X_test_clean = X_test.copy()

# Rellenamos distancia
X_train_clean['Shot_Distance_m'] = X_train_clean['Shot_Distance_m'].fillna(mediana_distancia)
X_test_clean['Shot_Distance_m'] = X_test_clean['Shot_Distance_m'].fillna(mediana_distancia)

# Rellenamos posición
X_train_clean['Playing_Position'] = X_train_clean['Playing_Position'].fillna(moda_posicion)
X_test_clean['Playing_Position'] = X_test_clean['Playing_Position'].fillna(moda_posicion)

# Verificamos que ya no existan nulos
print("\nNulos restantes en X_train_clean:", X_train_clean.isna().sum().sum())
print("Nulos restantes en X_test_clean:", X_test_clean.isna().sum().sum())

In [ ]:
# Buscamos filas donde el minuto sea mayor a 120 (límite fisico de un partido)
outliers_minuto = X_train_clean[X_train_clean['Minute'] > 120]

print("Filas con minutos atípicos en X_train:")
print(outliers_minuto[['Opponent', 'Minute']])

In [ ]:
# Reemplazamos cualquier minuto mayor a 120 por 120 en Train y Test
X_train_clean.loc[X_train_clean['Minute'] > 120, 'Minute'] = 120
X_test_clean.loc[X_test_clean['Minute'] > 120, 'Minute'] = 120

print("Minuto maximo en X_train_clean después del recorte:", X_train_clean['Minute'].max())